In [16]:
import csv
import json
import re

def load_dico():
    dico = set()
    with open('../resources/dico.csv', 'r', encoding='utf-8') as f:
        csv_reader = csv.reader(f, delimiter=',')
        for line in csv_reader:
            dico.add(line[0])
    return dico

def load_rules():
    with open('../resources/correct_rules.json', 'r', encoding='utf-8') as f:
        all_rules = json.load(f)
        rules = all_rules['rules']
        regex_rules = [re.compile(expr) for expr in all_rules['regex_rules']]
        return rules, regex_rules
    
def save_rules(rules, regex_rules):
    # print(rules)
    # print(regex_rules)
    regex_rules = [e.pattern for e in regex_rules]
    with open('../resources/correct_rules.json', mode='w', encoding='utf-8') as f:
        json.dump(
            {
                "rules": rules,
                "regex_rules": regex_rules
            },
            f,
            ensure_ascii=False,
            indent=2
        )


In [17]:
# on créé deux structures de données
# let set pour vérifier rapidement si un mot est dans le dico
# un dico taille_mots -> [mots] pour récupérer tous les mots d'une certaine longueur

dico = load_dico()
rules, regex_rules = load_rules()
words_by_size = {}
dico_list = list(dico)
dico_list.sort(key=len, reverse=True)
current_length = len(dico_list[0])
words_by_size[current_length] = [dico_list[0]]

for index, word in enumerate(dico_list):
    if len(word) == current_length:
        words_by_size[current_length].append(word)
        continue
    current_length = len(word)
    words_by_size[current_length] = [word]


dico = set(dico_list)
print(dico)

{'dim', 'gustave', 'amphitryon', 'pourlas', 'demoiselle', 'aus', 'compris', 'auxquelles', 'captifs', 'paris', 'solz', 'spectacle', 'soit', 'dapres', 'avertisse', 'exécution', 'heureusement', 'arsaces', 'amener', 'notoires', 'cedit', 'vaudra', 'louvart', 'rédigea', 'moindre', 'laissée', 'défait', 'avoir', 'rédenance', 'abandonner', 'béchet', 'chambault', 'balayage', 'morte', 'prests', 'timante', 'appuyer', 'lekaïn', 'emmagasiner', 'longtemps', 'incorporée', 'intéressées', 'contre', 'soyez', 'italiens', 'garde', 'égard', 'coté', 'blaise', 'connaissais', 'saisies', 'bonne', 'reconnaissance', 'nom', 'académie', 'tailleurs', 'receus', 'écrire', 'valeo', 'dud', 'voila', 'finir', 'operant', 'phraate', 'denier', 'mesnager', 'traiter', 'perdez', 'pair', 'expédition', 'noms', 'reprise', 'écaillée', 'usai', 'ira', 'gatent', 'fraces', 'balayeur', 'monsr', 'toutes', 'voudriez', 'doux', 'fovrnir', 'composition', 'conme', 'eriphile', 'presenté', 'humble', 'pretter', 'presq', 'marest', 'dus', 'ferney'

In [18]:
from Levenshtein import distance

def get_most_similar_words(word):
    possible_results = []
    all_words = words_by_size[len(word)] + words_by_size[len(word) + 1] + words_by_size[len(word) - 1]
    for word_to_compare in all_words:
        dist = distance(word, word_to_compare)
        if dist < 5:
            possible_results.append((word_to_compare, dist))
    possible_results.sort(key=lambda x: x[1])
    return possible_results

In [19]:
"assemblée" in dico


True

In [20]:
try:
    with open('../resources/test.txt', 'r', encoding='utf-8') as f:
        # TODO: proposer un mode automatique qui appliquera automatiquement toutes les corrections: à tester et à comparer après avec la vérité de terrain
        for line in f:
            for word in line.split(" "):
                if word.strip().lower() in dico:
                    continue
                else:                    
                    # TODO: rajouter si jamais le mot est dans les règles de correction stockées dans rules
                    # TODO: rajouter si jamais le mot est dans les règles regex avec expr.match stockées dans regex_rules
                    add_dict = input(f"Mot {word} invalide, ajouter au dictionnaire ? y/n")
                    if add_dict == 'y':
                        # TODO: ajouter au word_index
                        # pass
                        dico.add(word)
                    else:
                        res = get_most_similar_words(word)
                        question = ""
                        for index, correction in enumerate(res[:3]):
                            question += f"{index + 1}. {correction[0]} "
                        number = input("Entrez le numéro de la correction choisie (entrez 0 pour entrer votre propre correction): \n" + question)
                        try:
                            number = int(number)
                        except ValueError:
                            print("Bye bye")
                            raise KeyboardInterrupt()
                        bonne_correction = res[int(number)-1][0]
                        rules[word] = bonne_correction
                        # TODO sauvegarder la règle de correction avec la correction choisie
                        
                        # TODO: Ajouter une nouvelle règle: si le mot n'est pas ajouté dans le dictionnaire et n'est pas dans les corrections proposées
finally:
    save_rules(rules=rules,regex_rules=regex_rules)

Bye bye


KeyboardInterrupt: 